[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/dirac_waves/dirac_waves.ipynb)

# Spinor Waves on a Sphere

A wave of quaternions runs over a triangulated sphere: one quaternion, an even multivector, on each vertex, carried round the surface by the surface's Dirac operator. A pulse started at the north pole spreads as a ring and gathers again at the south pole, and the standing waves ring at whole-number frequencies. Every operator below is a *sparse extensor*: a linear map between fields on the mesh's vertices and faces, which couples each element to a few others, each coupling a multivector.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np
from IPython.display import Image, display

from numga import stack
from numga.sparse import spdiag
from examples.animation import save_animation
from examples.surfaces.dirac_waves import core, render
from examples.surfaces.dirac_waves.core import Even, Scalar, Vector, mv
from examples.surfaces.spin_transformations.core import icosphere

# The sphere's subdivisions, the time step, and the steps between frames.
SUBDIVISIONS = 3
INTERVAL = 0.04
EVERY = 2
# The pulse: its width, and the steps it runs, a little past the time to reach the far pole.
WIDTH = 0.25
STEPS = 84
# The standing waves: the frequencies shown, each held eight times its own, after the four constant
# fields of frequency zero; and the frames over each period.
LEVELS = 5
CONSTANT = 4
PHASES = 20

# The unit sphere: lengths in its radius, and times in its radius over the waves' speed.
sphere = icosphere(SUBDIVISIONS)


def bump(centre: Vector) -> Scalar:
    """A smooth bump on the sphere about the given point."""
    return ((sphere.vertices - centre).norm_squared() * (-0.5 / WIDTH**2)).exp()   # Scalar[V]

## 1. The Dirac operator

A field is an extensor with one element per vertex or face: `sphere.vertices` is `Vector[V]`, and the wave's field on the vertices is `Even[V]`. An operator between fields is a sparse extensor: for each output element, a few couplings to input elements, each a multivector. Times a field, each coupling multiplies its input element into its output element, and the products are summed. `D` couples each face to its three corners through vectors: the edge each corner faces, over minus twice the face's area. A vector times a quaternion is odd, so `D` takes a quaternion field on the vertices to an odd field on the faces. `~D` is its reverse: reversing a product turns it around, so every coupling is reversed and runs the other way, from a face to its corners.

Below is a bump of the xy plane at the north pole, beside the face field `D` makes of it. Red, green and blue are the vertex field's planes yz, zx and xy, together with the face field's directions x, y and z.

In the quaternionic matrix notation of Crane, Pinkall and Schröder the operator reads as $D_{ij} = -\tfrac{1}{2\mathcal{A}_i} e_j$.

In [ ]:
# diagonal operators: the face areas, and the vertex areas and their inverses
M2 = spdiag(sphere.triangle_areas)                                     # [F, F] Scalar
M2i = spdiag(1 / sphere.triangle_areas)                                # [F, F] Scalar
M0 = spdiag(sphere.vertex_areas)                                       # [V, V] Scalar
M0i = spdiag(1 / sphere.vertex_areas)                                  # [V, V] Scalar

# each face takes each corner by the edge it faces, over minus twice the face's area
D = M2i * sphere.at_corners(sphere.triangle_edges) * -0.5              # [F, V] Vector

start = bump(mv.z) * mv.xy                                              # Bivector[V]

print(D.gatype, D.shape)
print((~D).gatype, (~D).shape)
render.still(sphere, start, D * start);

## 2. A pulse

The wave alternates the two operators: the face field moves on by `D` of the vertex field, then the vertex field moves back by `~D` of the face field, weighted by the face areas and divided by the vertex areas. This leapfrog keeps the two fields' energy, the area-weighted squared size of both. The pulse spreads from the north pole as a ring, and half a turn round the sphere later, at time π, it gathers at the south pole.

In [ ]:
def spread(start: Even, interval: float, count: int):
    """A wave from the given vertex field: the vertex field at each step and the face field half a
    step after it."""
    vertices, faces = start, D * start * 0.0                            # Even[V], Odd[F]
    for _ in range(count):
        # the face field moved on by the Dirac operator of the vertex field
        faces = faces + D * vertices * interval                         # Odd[F]
        yield vertices, faces
        # the vertex field moved back by the reverse of the face field, weighted by the areas
        vertices = vertices - M0i * (~D * (M2 * faces)) * interval      # Even[V]


states = list(spread(start, INTERVAL, STEPS))
frames = states[::EVERY]
pulse_vertices = stack([vertices for vertices, _ in frames])            # [frames] Even[V]
pulse_faces = stack([faces for _, faces in frames])                     # [frames] Odd[F]
display(Image(filename=save_animation(render.animate(sphere, pulse_vertices[None], pulse_faces[None], 1), "dirac_waves_pulse", 80)))

## 3. Standing waves

Twice over a step is `Q = ~D * M2 * D`, from vertex fields to vertex fields, its couplings even. An eigensolver has to know what a linear map acts on, and the product with an open type tells it: `Q * Even` is a map of quaternion fields into themselves, solved against the vertex areas, `M0 * Even`. Its eigenvalues are the standing waves' frequencies squared. On the unit sphere these are the whole numbers, each held by eight waves per unit of frequency, after four constant fields at zero.

Of each frequency's waves, the one drawn is the part of a bump on the equator they hold, so every frequency is drawn the same way round. Over its period the vertex field swings as a cosine and the face field, `D` of it over the frequency, as a sine.

In [ ]:
# how many waves up to each frequency: the constant ones, then eight per unit of each frequency
sizes = CONSTANT + 8 * np.cumsum(np.arange(LEVELS + 1))                 # [levels + 1]
Q = ~D * M2 * D                                                         # [V, V] Even
squared, waves = (Q * Even).eigh(M0 * Even, int(sizes[-1]))             # [modes] Scalar, [modes] Even[V]

# of each frequency's waves, the part of a bump on the equator they hold, in all three planes
probe = bump(mv.x) * (mv.yz + mv.zx + mv.xy)                            # Even[V]
shares = [waves[low:high] for low, high in zip(sizes[:-1], sizes[1:])]
chosen = stack([(share * (M0 * share.reverse().scalar_product(probe)).sites.sum()).sum(axis=0)
                for share in shares])                                   # [levels] Even[V]

# each over its period: the vertex field a cosine, the face field, D of it over the frequency, a sine
frequencies = np.arange(1, LEVELS + 1)                                  # [levels]
phases = np.linspace(0.0, 2 * np.pi, PHASES, endpoint=False)            # [phases]
mode_vertices = chosen[:, None] * np.cos(phases)                        # [levels, phases] Even[V]
mode_faces = (D * chosen * (1 / frequencies))[:, None] * np.sin(phases)  # [levels, phases] Odd[F]

print(Q.gatype, Q.shape)
print((Q * Even).gatype)
display(Image(filename=save_animation(render.animate(sphere, mode_vertices, mode_faces, LEVELS), "dirac_waves_modes", 50)))

In [ ]:
# checks
# The operator laid out here is the core's.
np.testing.assert_allclose((core.dirac(sphere)[0] - D).cells.kernel, 0.0, atol=1e-12)
# Weighted by the areas, D and its reverse pair a face field and a vertex field the same.
paired = (M2 * mode_faces[0, PHASES // 4].reverse().scalar_product(D * probe)).sites.sum()
reversed_pairing = (~D * (M2 * mode_faces[0, PHASES // 4])).reverse().scalar_product(probe).sites.sum()
np.testing.assert_allclose(paired.to_array(), reversed_pairing.to_array(), rtol=1e-12)
# The leapfrog keeps the energy: the vertex field's area-weighted squared size, and the face field's
# paired across each step.
energies = stack([(M0 * vertices.scalar_norm_squared()).sites.sum() + (M2 * before.reverse().scalar_product(after)).sites.sum()
                  for (vertices, after), (_, before) in zip(states[1:], states)]).to_array()
np.testing.assert_allclose(energies, energies[0], rtol=1e-12)
# Half a turn round the sphere on, the pulse has gathered at the south pole.
density = (M0 * states[int(round(np.pi / INTERVAL))][0].scalar_norm_squared()).to_array()
assert density[(sphere.vertices | mv.z) < -0.9].sum() > 0.5 * density.sum()
# The frequencies are the whole numbers, each held eight times its own, after the constant fields,
# up to the mesh's resolution.
measured = squared.to_array()
np.testing.assert_allclose(measured[:CONSTANT], 0.0, atol=1e-9)
expected = np.repeat(frequencies, 8 * frequencies)
np.testing.assert_allclose(np.sqrt(measured[CONSTANT:]), expected, rtol=0.03)